# Phase 5 - Step 1: Legal Statute Identification (LSI) data exploration

**Task:** read a legal case, output the **set** of statute IDs that apply (multi-label: one case can have
several correct IDs). Metric: **macro-F1 over statute IDs** (every statute counts equally, so rare statutes matter
as much as common ones); secondary micro-F1 and subset accuracy (exact set match).

What this notebook finds out before we design any model (no GPU needed):

1. download `Phase_5_LSI/` from Hugging Face (skipped if already on disk)
2. real file format (the doc only shows an example) + data checks: duplicates, conflicting labels, empty texts, leaks between splits, labels hidden in test
3. **label space**: how many statute IDs, which IDs, labels per case
4. **label frequency** (long tail -> rare statutes) and train vs dev shift
5. **label co-occurrence** (which statutes appear together)
6. scorer (macro-F1 / micro-F1 / subset accuracy, same definitions as sklearn) + **no-reading baselines**
7. writing style (cased vs lower-case) and lengths in words and **bpe32k tokens** -> what fits in 1536 tokens, time per epoch
8. **are statutes named in the text?** ("Section 302 ...") and where in the case they appear
9. **which part of the case carries the signal?** a tiny word-based classifier (Naive Bayes, one per statute) on first 512 / last 512 / first 256 + last 256 / whole case
10. words most typical for the most common statutes (what each ID probably means)

Results are cached in `experiments/phase5/p5_01_data/` (re-running loads them).


In [1]:
import json, re, sys, hashlib, collections, itertools, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# project root = first folder upwards that contains .gitignore (same as all earlier notebooks)
ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from ner_utils import CHAR_MAP          # the Phase-1/2 character cleaning table

RAW_DIR = ROOT / "data" / "raw"
P5 = RAW_DIR / "Phase_5_LSI"
OUT = ROOT / "experiments" / "phase5" / "p5_01_data"
OUT.mkdir(parents=True, exist_ok=True)
TOK_PATH = ROOT / "data" / "tokenizer" / "bpe32k.json"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download the Phase 5 files (skipped if already on disk)

Expected sizes (from the Hugging Face file list, 24 Oct): train.json 299 MB, dev.json 79 MB, test.json 98 MB.
The repository is gated: you must be logged in (`hf auth login`, done in earlier phases).


In [2]:
FILES = {"train": P5 / "Train" / "train.json", "dev": P5 / "Train" / "dev.json", "test": P5 / "Test" / "test.json"}
if all(p.exists() for p in FILES.values()):
    print("Phase 5 data already downloaded -> skipping download")
else:
    from huggingface_hub import snapshot_download
    snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns="Phase_5_LSI/*", local_dir=str(RAW_DIR))
for p in sorted(P5.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(P5)):25s} {p.stat().st_size / 1e6:9.2f} MB")
extra = [p for p in P5.rglob("*") if p.is_file() and p not in FILES.values()]
if extra:
    print("NOTE: files not used by this notebook:", [str(p.relative_to(P5)) for p in extra])

Fetching 3 files: 100%|██████████| 3/3 [00:20<00:00,  6.98s/it]

Test\test.json                98.06 MB
Train\dev.json                79.38 MB
Train\train.json             299.40 MB


## 2. Load the files and look at the real format

The doc shows `{"id", "text", "labels": [1, 7, 23]}`, but past phases taught us to check instead of trusting:

- the file can be a **list** of records or a **dict** `{id: record}`
- `text` can be one string or a **list of sentences/paragraphs** (then joined with a space; we count the pieces)
- the **label key** = first of `labels, label, statutes, statute_ids, y` that is present; labels may be ints or strings
- Phase 4's test.json secretly contained labels -> if test has labels here too, we **drop them on load** and never use them


In [3]:
LABEL_KEYS = ["labels", "label", "statutes", "statute_ids", "statute", "y", "targets"]
TEXT_KEYS = ["text", "document", "doc", "case_text", "facts", "content", "judgment"]

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def to_doc_list(data):
    # a list of records, or a dict {id: record}
    if isinstance(data, list):
        return [dict(d) for d in data]
    if isinstance(data, dict):
        out = []
        for k, v in data.items():
            d = dict(v) if isinstance(v, dict) else {"text": v}
            d.setdefault("id", k)
            out.append(d)
        return out
    raise TypeError(f"unexpected JSON type {type(data)}")

def find_key(record, candidates):
    return next((k for k in candidates if k in record), None)

def as_label_list(lab):
    # labels -> list (a single value becomes a one-element list; None stays None)
    if lab is None:
        return None
    return list(lab) if isinstance(lab, (list, tuple)) else [lab]

SPLITS, FORMAT, RAW_LABEL_TYPES = {}, {}, {}
for name, path in FILES.items():
    raw = read_json(path)
    recs = to_doc_list(raw)
    first = recs[0]
    tkey, lkey = find_key(first, TEXT_KEYS), find_key(first, LABEL_KEYS)
    assert tkey, f"{name}: no text field among {list(first.keys())}"
    pieces = [len(d[tkey]) if isinstance(d[tkey], list) else 1 for d in recs]
    docs = []
    for d in recs:
        t = d[tkey]
        t = " ".join(map(str, t)) if isinstance(t, list) else str(t)
        docs.append({"id": str(d["id"]), "text": t, "labels": as_label_list(d.get(lkey)) if lkey else None})
    RAW_LABEL_TYPES[name] = sorted({type(x).__name__ for d in docs if d["labels"] for x in d["labels"]})
    FORMAT[name] = {"file_type": type(raw).__name__, "records": len(recs), "keys": list(first.keys()),
                    "text_key": tkey, "text_type": type(first[tkey]).__name__,
                    "pieces_per_text (mean)": round(float(np.mean(pieces)), 1),
                    "label_key": lkey, "label_value_types": RAW_LABEL_TYPES[name]}
    SPLITS[name] = docs
    print(f"===== {name}: {len(recs):,} records | {FORMAT[name]}")
    d0 = docs[0]
    print(f"  first id={d0['id']!r} labels={d0['labels']} | text starts: {d0['text'][:250]!r}\n")

TEST_HAD_LABELS = any(d["labels"] for d in SPLITS["test"])
if TEST_HAD_LABELS:
    print("WARNING: test.json contains labels -> dropped now, never used (same as Phase 4)")
    for d in SPLITS["test"]:
        d["labels"] = None

===== train: 42,750 records | {'file_type': 'list', 'records': 42750, 'keys': ['id', 'text', 'labels'], 'text_key': 'text', 'text_type': 'str', 'pieces_per_text (mean)': 1.0, 'label_key': 'labels', 'label_value_types': ['int']}
  first id='1000008' labels=[69, 9, 3] | text starts: '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of we'

===== dev: 10,181 records | {'file_type': 'list', 'records': 10181, 'keys': ['id', 'text', 'labels'], 'text_key': 'text', 'text_type': 'str', 'pieces_per_text (mean)': 1.0, 'label_key': 'labels', 'label_value_types': ['int']}
  first id='1000139' labels=[5] | text starts: 'Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the prosecution relied on the evidence o

**Label IDs must be integers** in the submission. If the file stores them as strings (e.g. `"Section 302"`), the cell
below shows it, and we will need an ID mapping. If they are numeric strings (`"7"`) they are converted to ints.


In [4]:
def to_int_label(x):
    if isinstance(x, bool):
        raise ValueError(x)
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float) and x.is_integer():
        return int(x)
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        return int(x.strip())
    raise ValueError(x)

bad = collections.Counter()
for name in ("train", "dev"):
    for d in SPLITS[name]:
        labs = []
        for x in d["labels"] or []:
            try:
                labs.append(to_int_label(x))
            except ValueError:
                bad[str(x)] += 1
        d["labels_raw_len"] = len(d["labels"] or [])
        d["labels"] = sorted(set(labs))          # a set: no duplicates, sorted
if bad:
    print("NON-INTEGER labels found (need a mapping before modelling):", bad.most_common(20))
else:
    print("all train/dev labels are integers (or integer strings) -> usable directly as statute IDs")
dup_in_list = sum(d["labels_raw_len"] != len(d["labels"]) for n in ("train", "dev") for d in SPLITS[n])
print("cases whose label list contained the same ID twice:", dup_in_list)

all train/dev labels are integers (or integer strings) -> usable directly as statute IDs
cases whose label list contained the same ID twice: 4


## 3. Data checks

- duplicate ids, empty / very short texts, cases with **no** label
- line breaks (structure), non-ASCII characters (Phase 1 trained on cleaned ASCII text), PDF signature stamps
- **duplicate texts** inside and across splits (a train case that reappears in dev/test inflates scores;
  the same text with **different label sets** = label noise)


In [5]:
def text_key(t):
    # whitespace-insensitive, case-insensitive fingerprint of a document (same as Phase 4)
    return hashlib.md5(re.sub(r"\s+", " ", t).strip().lower().encode()).hexdigest()

rows, by_hash = [], collections.defaultdict(list)
for name, docs in SPLITS.items():
    ids = [d["id"] for d in docs]
    n_chars = sum(len(d["text"]) for d in docs)
    non_ascii = sum(sum(ord(c) > 127 for c in d["text"]) for d in tqdm(docs, desc=f"checking {name}", leave=False))
    for d in docs:
        by_hash[text_key(d["text"])].append((name, d["id"], tuple(d["labels"]) if d["labels"] is not None else None))
    rows.append({
        "split": name, "docs": len(docs), "dup_ids": len(ids) - len(set(ids)),
        "no_labels": sum(d["labels"] is not None and len(d["labels"]) == 0 for d in docs),
        "empty": sum(not d["text"].strip() for d in docs),
        "<100 words": sum(len(d["text"].split()) < 100 for d in docs),
        "docs_with_newlines_%": round(100 * np.mean(["\n" in d["text"] for d in docs]), 1),
        "non_ascii_%": round(100 * non_ascii / max(n_chars, 1), 3),
        "signature_stamps": sum("signature not verified" in d["text"].lower() for d in docs),
    })
checks = pd.DataFrame(rows).set_index("split")
display(checks)

dup_groups = [v for v in by_hash.values() if len(v) > 1]
pair_counts = collections.Counter(tuple(sorted({s for s, _, _ in v})) for v in dup_groups)
conflicting = [v for v in dup_groups if len({l for _, _, l in v if l is not None}) > 1]
print(f"duplicate texts: {len(dup_groups)} groups ({sum(len(v) - 1 for v in dup_groups)} extra copies); "
      f"splits they span: { {'&'.join(k): c for k, c in pair_counts.items()} }")
print(f"  duplicate groups with CONFLICTING label sets: {len(conflicting)}")
for v in conflicting[:3]:
    print("  example conflict:", v[:3])
ids_shared = {a + "&" + b: len({d["id"] for d in SPLITS[a]} & {d["id"] for d in SPLITS[b]})
              for a, b in [("train", "dev"), ("train", "test"), ("dev", "test")]}
print("ids shared between splits:", ids_shared)

,docs,dup_ids,no_labels,empty,<100 words,docs_with_newlines_%,non_ascii_%,signature_stamps
split,,,,,,,,
train,42750,0,0,0,4845,0.0,0.005,274
dev,10181,0,0,0,1165,0.0,0.009,91
test,13019,0,0,0,1434,0.0,0.011,128


duplicate texts: 481 groups (807 extra copies); splits they span: {'train': 219, 'test&train': 86, 'dev&test&train': 23, 'dev&train': 58, 'dev&test': 39, 'dev': 23, 'test': 33}
  duplicate groups with CONFLICTING label sets: 101
  example conflict: [('train', '148520751', (5, 40, 43, 45, 83, 96, 97)), ('train', '188537125', (5, 77, 97)), ('train', '190392407', (5, 39, 43, 45, 63, 83, 97))]
  example conflict: [('train', '75580037', (18,)), ('train', '83171322', (5, 21, 39, 47, 53)), ('train', '113734007', (5, 43, 45, 61, 81, 83, 97, 99))]
  example conflict: [('train', '80165899', (5, 10, 34)), ('train', '139487033', (12, 13, 14, 21, 39, 45, 46, 47, 53, 63, 78)), ('train', '163217338', (12, 13, 14))]
ids shared between splits: {'train&dev': 0, 'train&test': 0, 'dev&test': 0}


## 4. Label space and labels per case

- how many **different statute IDs** exist (= the size of the classification head) and whether they are 0..L-1
- IDs that appear in dev but never in train (a model cannot learn them)
- **label cardinality** = average number of statutes per case; **density** = cardinality / number of IDs
  (= the share of "1"s in the case x statute matrix -> how unbalanced each yes/no decision is)


In [6]:
LAB_SPLITS = ["train", "dev"]
FREQ = {n: collections.Counter(l for d in SPLITS[n] for l in d["labels"]) for n in LAB_SPLITS}
ALL_IDS = sorted(set(FREQ["train"]) | set(FREQ["dev"]))
L = len(ALL_IDS)
print(f"statute IDs: {L} different (train {len(FREQ['train'])}, dev {len(FREQ['dev'])}); "
      f"min {min(ALL_IDS)}, max {max(ALL_IDS)}")
missing = sorted(set(range(min(ALL_IDS), max(ALL_IDS) + 1)) - set(ALL_IDS))
print(f"IDs missing inside the range {min(ALL_IDS)}..{max(ALL_IDS)}: {len(missing)} {missing[:20]}")
dev_only = sorted(set(FREQ["dev"]) - set(FREQ["train"]))
train_only = sorted(set(FREQ["train"]) - set(FREQ["dev"]))
print(f"IDs in dev but NOT in train (unlearnable): {len(dev_only)} {dev_only[:20]}")
print(f"IDs in train but not in dev: {len(train_only)} {train_only[:20]}")

card = {}
for n in LAB_SPLITS:
    k = np.array([len(d["labels"]) for d in SPLITS[n]])
    card[n] = {"cases": len(k), "labels/case mean": round(k.mean(), 3), "median": int(np.median(k)),
               "max": int(k.max()), "density %": round(100 * k.mean() / L, 3),
               **{f"{i} labels %": round(100 * (k == i).mean(), 1) for i in range(0, 5)},
               ">=5 labels %": round(100 * (k >= 5).mean(), 1)}
display(pd.DataFrame(card).T.rename_axis("labels per case"))

statute IDs: 100 different (train 100, dev 100); min 0, max 99
IDs missing inside the range 0..99: 0 []
IDs in dev but NOT in train (unlearnable): 0 []
IDs in train but not in dev: 0 []


,cases,labels/case mean,median,max,density %,0 labels %,1 labels %,2 labels %,3 labels %,4 labels %,>=5 labels %
labels per case,,,,,,,,,,,
train,42750.0,3.705,3.0,31.0,3.705,0.0,21.4,17.7,16.5,13.5,30.9
dev,10181.0,3.916,3.0,28.0,3.916,0.0,15.2,18.7,17.9,15.1,33.2


## 5. Label frequency: the long tail (rare statutes)

Macro-F1 = the plain average of the per-statute F1 scores. A statute seen in 30 training cases counts **as much** as one
seen in 10,000. So the rare statutes decide a big part of the score. The buckets below show how many statutes are
head / medium / tail, and what share of all label occurrences they cover.


In [7]:
tr_n = len(SPLITS["train"]); dv_n = len(SPLITS["dev"])
freq = pd.DataFrame({"train": pd.Series(FREQ["train"]), "dev": pd.Series(FREQ["dev"])}).reindex(ALL_IDS).fillna(0).astype(int)
freq["train %cases"] = (100 * freq["train"] / tr_n).round(2)
freq["dev %cases"] = (100 * freq["dev"] / dv_n).round(2)
freq["pos_weight (neg/pos)"] = ((tr_n - freq["train"]) / freq["train"].replace(0, np.nan)).round(1)
freq = freq.sort_values("train", ascending=False)
freq["train rank"] = np.arange(1, len(freq) + 1)
print("most frequent statute IDs:")
display(freq.head(15))
print("least frequent statute IDs:")
display(freq.tail(10))

BUCKETS = [(1000, np.inf, ">=1000"), (300, 1000, "300-999"), (100, 300, "100-299"), (30, 100, "30-99"),
           (10, 30, "10-29"), (1, 10, "1-9"), (0, 1, "0 (dev only)")]
tot = freq["train"].sum()
brows = []
for lo, hi, name in BUCKETS:
    m = (freq["train"] >= lo) & (freq["train"] < hi)
    brows.append({"train cases per statute": name, "statutes": int(m.sum()),
                  "share of macro-F1 weight %": round(100 * m.sum() / L, 1),
                  "share of label occurrences %": round(100 * freq.loc[m, "train"].sum() / tot, 1)})
display(pd.DataFrame(brows).set_index("train cases per statute"))
top10 = freq["train"].head(10).sum() / tot
print(f"the 10 most frequent statutes = {100 * top10:.1f}% of all train label occurrences "
      f"but only {100 * 10 / L:.1f}% of the macro-F1 weight")

# train vs dev shift: Spearman rank correlation of the frequencies (numpy only)
def spearman(a, b):
    ra, rb = pd.Series(a).rank().values, pd.Series(b).rank().values
    return float(np.corrcoef(ra, rb)[0, 1])
print(f"train vs dev frequency (share of cases): Spearman {spearman(freq['train %cases'], freq['dev %cases']):.3f}")
big_shift = freq[(freq["dev"] >= 10)].assign(ratio=lambda f: (f["dev %cases"] / f["train %cases"].replace(0, np.nan)).round(2))
print("statutes whose share changes most between train and dev (dev >= 10 cases):")
bs = big_shift.sort_values("ratio")[["train", "dev", "train %cases", "dev %cases", "ratio"]]
display(bs if len(bs) <= 10 else pd.concat([bs.head(5), bs.tail(5)]))

most frequent statute IDs:


,train,dev,train %cases,dev %cases,pos_weight (neg/pos),train rank
5,11730,2936,27.44,28.84,2.6,1
34,10380,2596,24.28,25.50,3.1,2
97,7880,1970,18.43,19.35,4.4,3
43,7728,1932,18.08,18.98,4.5,4
39,7120,1779,16.65,17.47,5.0,5
12,4324,1086,10.11,10.67,8.9,6
13,4136,1033,9.67,10.15,9.3,7
10,4134,1033,9.67,10.15,9.3,8
77,4046,1015,9.46,9.97,9.6,9
94,3835,958,8.97,9.41,10.1,10


least frequent statute IDs:


,train,dev,train %cases,dev %cases,pos_weight (neg/pos),train rank
15,336,84,0.79,0.83,126.2,91
27,329,83,0.77,0.82,128.9,92
74,319,80,0.75,0.79,133.0,93
50,318,80,0.74,0.79,133.4,94
59,310,86,0.73,0.84,136.9,95
66,309,78,0.72,0.77,137.3,96
92,309,77,0.72,0.76,137.3,97
23,304,80,0.71,0.79,139.6,98
25,287,72,0.67,0.71,148.0,99
84,272,68,0.64,0.67,156.2,100


,statutes,share of macro-F1 weight %,share of label occurrences %
train cases per statute,,,
>=1000,43,43.0,81.8
300-999,55,55.0,17.9
100-299,2,2.0,0.4
30-99,0,0.0,0.0
10-29,0,0.0,0.0
1-9,0,0.0,0.0
0 (dev only),0,0.0,0.0


the 10 most frequent statutes = 41.2% of all train label occurrences but only 10.0% of the macro-F1 weight
train vs dev frequency (share of cases): Spearman 0.996
statutes whose share changes most between train and dev (dev >= 10 cases):


,train,dev,train %cases,dev %cases,ratio
30,3009,405,7.04,3.98,0.57
80,573,141,1.34,1.38,1.03
95,375,93,0.88,0.91,1.03
48,585,145,1.37,1.42,1.04
49,380,95,0.89,0.93,1.04
40,543,152,1.27,1.49,1.17
53,1276,363,2.98,3.57,1.20
65,351,105,0.82,1.03,1.26
96,1973,606,4.62,5.95,1.29
11,615,232,1.44,2.28,1.58


## 6. Label co-occurrence (which statutes come together)

- number of **different label sets** (if few sets cover most cases, predicting a whole set is an option)
- most frequent pairs with `P(B | A)` (if A is present, how often is B present too) and **lift** = P(A and B) / (P(A) P(B))
  (lift 1 = independent; lift >> 1 = they appear together far more than by chance, e.g. a punishment section and its definition)


In [8]:
sets = collections.Counter(tuple(d["labels"]) for d in SPLITS["train"])
cov = np.cumsum(sorted(sets.values(), reverse=True)) / tr_n
print(f"different label sets in train: {len(sets):,} for {tr_n:,} cases; "
      f"top 10 sets cover {100 * cov[min(9, len(cov) - 1)]:.1f}%, top 100 cover {100 * cov[min(99, len(cov) - 1)]:.1f}%")
display(pd.DataFrame([(list(s), c, round(100 * c / tr_n, 2)) for s, c in sets.most_common(10)],
                     columns=["label set", "cases", "%"]))
dev_sets_seen = np.mean([tuple(d["labels"]) in sets for d in SPLITS["dev"]])
print(f"dev cases whose exact label set also occurs in train: {100 * dev_sets_seen:.1f}%")

pairs = collections.Counter(p for d in SPLITS["train"] for p in itertools.combinations(d["labels"], 2))
prow = []
for (a, b), c in pairs.most_common(15):
    fa, fb = FREQ["train"][a], FREQ["train"][b]
    prow.append({"A": a, "B": b, "cases together": c, "P(B|A)": round(c / fa, 3), "P(A|B)": round(c / fb, 3),
                 "lift": round(c * tr_n / (fa * fb), 2)})
print("most frequent statute pairs (train):")
display(pd.DataFrame(prow))
# strongest dependencies among reasonably frequent statutes
strong = []
for (a, b), c in pairs.items():
    if c >= 30:
        fa, fb = FREQ["train"][a], FREQ["train"][b]
        strong.append({"A": a, "B": b, "cases together": c, "P(B|A)": round(c / fa, 3), "P(A|B)": round(c / fb, 3),
                       "lift": round(c * tr_n / (fa * fb), 2)})
if strong:
    print("pairs with the highest lift (>= 30 cases together):")
    display(pd.DataFrame(strong).sort_values("lift", ascending=False).head(10))

different label sets in train: 16,191 for 42,750 cases; top 10 sets cover 10.5%, top 100 cover 31.2%


,label set,cases,%
0,[34],909,2.13
1,[5],796,1.86
2,[61],453,1.06
3,"[5, 34]",425,0.99
4,[39],359,0.84
5,"[20, 28]",342,0.80
6,[10],321,0.75
7,[54],311,0.73
8,[77],290,0.68
9,[94],271,0.63


dev cases whose exact label set also occurs in train: 57.8%
most frequent statute pairs (train):


,A,B,cases together,P(B|A),P(A|B),lift
0,5,34,3644,0.311,0.351,1.28
1,12,13,3511,0.812,0.849,8.39
2,43,97,3385,0.438,0.430,2.38
3,5,43,3271,0.279,0.423,1.54
4,13,14,2831,0.684,0.741,7.66
5,5,39,2828,0.241,0.397,1.45
6,12,14,2672,0.618,0.700,6.92
7,34,39,2596,0.250,0.365,1.50
8,5,97,2424,0.207,0.308,1.12
9,34,35,2174,0.209,0.689,2.84


pairs with the highest lift (>= 30 cases together):


,A,B,cases together,P(B|A),P(A|B),lift
335,20,28,451,0.926,0.749,65.76
9,15,16,277,0.824,0.417,53.00
51,29,49,222,0.447,0.584,50.25
45,49,50,142,0.374,0.447,50.24
52,29,50,164,0.330,0.516,44.36
170,64,86,228,0.361,0.628,42.49
208,23,25,80,0.263,0.279,39.20
527,29,36,245,0.493,0.431,37.04
295,21,47,442,0.646,0.503,31.43
296,21,53,630,0.921,0.494,30.86


## 7. Scorer and no-reading baselines

Same definitions as `sklearn.metrics.f1_score` on a binary case x statute matrix (own numpy code; sklearn is not
in the .venv). For one statute: TP = predicted and correct, FP = predicted but wrong, FN = missed;
`F1 = 2 TP / (2 TP + FP + FN)` (0 if the statute never appears in gold or prediction).

- **macro-F1** = mean of the per-statute F1. The doc says _"the evaluator derives the label dimension from the reference
  data"_ -> we average over the statutes present in the **gold** labels (`macro_f1`). We also print the sklearn version
  where the label set = gold ∪ predicted (`macro_f1_union`): a predicted statute that never appears in gold adds an F1 of 0.
- **micro-F1** = one F1 from the summed TP / FP / FN of all statutes (dominated by frequent statutes)
- **subset accuracy** = share of cases whose predicted set is exactly the gold set

Baselines that do not read the text show the floor: empty prediction, "always the most frequent statute",
"always the k most frequent", random guessing with the train frequencies.


In [9]:
def lsi_scores(gold, pred, label_set=None):
    # gold, pred: lists of label lists (one per case). Returns macro/micro-F1, subset accuracy and per-label table.
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    exact = 0
    for g, p in zip(gold, pred):
        g, p = set(g), set(p)
        exact += g == p
        for l in g & p: tp[l] += 1
        for l in p - g: fp[l] += 1
        for l in g - p: fn[l] += 1
    gold_labels = sorted({l for g in gold for l in g})
    union_labels = sorted(set(gold_labels) | {l for p in pred for l in p})
    def f1(l):
        d = 2 * tp[l] + fp[l] + fn[l]
        return 2 * tp[l] / d if d else 0.0
    per = {l: f1(l) for l in union_labels}
    labels = gold_labels if label_set is None else label_set
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    return {"macro_f1": round(float(np.mean([per.get(l, 0.0) for l in labels])), 4),
            "macro_f1_union": round(float(np.mean([per[l] for l in union_labels])), 4),
            "micro_f1": round(2 * TP / max(2 * TP + FP + FN, 1), 4),
            "micro_p": round(TP / max(TP + FP, 1), 4), "micro_r": round(TP / max(TP + FN, 1), 4),
            "subset_acc": round(exact / len(gold), 4), "labels/case predicted": round(np.mean([len(set(p)) for p in pred]), 2),
            "per_label": per}

# sanity check against hand-computed numbers
g = [[1, 2], [2], [3]]; p = [[1], [2, 3], []]
s = lsi_scores(g, p)
# label 1: TP1 -> F1 1.0 ; label 2: TP1 FN1 -> 2/3 ; label 3: FP1 FN1 -> 0 ; macro = (1 + 2/3 + 0) / 3 = 0.5556
# micro: TP 2, FP 1, FN 2 -> 4 / 7 = 0.5714 ; subset accuracy: only case 2? no -> {2} vs {2,3} differ -> 0 / 3
assert abs(s["macro_f1"] - 0.5556) < 1e-4 and abs(s["micro_f1"] - 0.5714) < 1e-4 and s["subset_acc"] == 0.0
print("scorer sanity check OK")

gold_dev = [d["labels"] for d in SPLITS["dev"]]
order = list(freq.index)                    # statutes sorted by train frequency
k_med = max(1, int(round(card["train"]["labels/case mean"])))
rng = np.random.default_rng(0)
p_train = (freq["train"] / tr_n).reindex(ALL_IDS).values
BASE = {
    "empty prediction": [[] for _ in gold_dev],
    "always the most frequent statute": [order[:1] for _ in gold_dev],
    f"always the {k_med} most frequent": [order[:k_med] for _ in gold_dev],
    "always the 5 most frequent": [order[:5] for _ in gold_dev],
    "random with train frequencies": [[l for l, q in zip(ALL_IDS, rng.random(L) < p_train) if q] for _ in gold_dev],
}
base_tbl = pd.DataFrame({k: {m: v for m, v in lsi_scores(gold_dev, p).items() if m != "per_label"} for k, p in BASE.items()}).T
display(base_tbl.rename_axis("dev baseline (no reading)"))

scorer sanity check OK


,macro_f1,macro_f1_union,micro_f1,micro_p,micro_r,subset_acc,labels/case predicted
dev baseline (no reading),,,,,,,
empty prediction,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00
always the most frequent statute,0.0045,0.0045,0.1173,0.2884,0.0736,0.0028,1.00
always the 4 most frequent,0.0150,0.0150,0.2341,0.2317,0.2366,0.0000,4.00
always the 5 most frequent,0.0179,0.0179,0.2471,0.2203,0.2813,0.0000,5.00
random with train frequencies,0.0389,0.0389,0.1004,0.1032,0.0978,0.0004,3.71


## 8. Writing style and lengths (words, characters)

Phase 4 surprise: train was cased modern text, dev/test lower-case old text. We check the same here:
share of cases with < 1% upper-case letters, and share with line breaks.


In [10]:
def style(t):
    letters = sum(c.isalpha() for c in t[:20000])
    upper = sum(c.isupper() for c in t[:20000])
    return "lower" if upper < 0.01 * max(letters, 1) else "cased"

STYLE = {n: collections.Counter(style(d["text"]) for d in tqdm(docs, desc=f"style {n}", leave=False)) for n, docs in SPLITS.items()}
display(pd.DataFrame(STYLE).fillna(0).astype(int).T.assign(**{"lower %": lambda f: (100 * f.get("lower", 0) / f.sum(axis=1)).round(1)}))

def pct_row(x):
    x = np.asarray(x)
    return {"mean": round(float(x.mean()), 1), "p10": int(np.percentile(x, 10)), "p50": int(np.percentile(x, 50)),
            "p90": int(np.percentile(x, 90)), "p99": int(np.percentile(x, 99)), "max": int(x.max())}

WORDS = {n: [len(d["text"].split()) for d in docs] for n, docs in SPLITS.items()}
display(pd.DataFrame({n: pct_row(w) for n, w in WORDS.items()}).T.rename_axis("words per document"))

,cased,lower,lower %
train,42727,23,0.1
dev,10179,2,0.0
test,13016,3,0.0


,mean,p10,p50,p90,p99,max
words per document,,,,,,
train,1124.2,90.0,430.0,2766.0,8142.0,86858.0
dev,1248.5,90.0,543.0,2957.0,8855.0,69230.0
test,1211.7,94.0,532.0,2811.0,8306.0,134848.0


## 9. Lengths in bpe32k tokens (cached)

Each case is cleaned exactly like the Phase-1 training text (signature stamps removed, special characters mapped to
ASCII, other non-ASCII dropped, all whitespace -> single spaces; **no lower-casing here**, that decision comes after
the style check), then tokenised with my bpe32k tokenizer.

From the counts: share of cases that fit in one 1536-token window (Phase-1 context), windows per case
(1536 tokens, stride 1024), and **rough time per epoch** for each way of reading a case, using speeds measured in
Phase 4 (A2 full fine-tuning on last-1536 inputs: ~26k tokens/s training, ~90k tokens/s inference).


In [11]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(TOK_PATH))
print("tokenizer:", TOK_PATH.relative_to(ROOT), "| vocab", tok.get_vocab_size())

SIG_RE = re.compile(r"Signature Not Verified.{0,300}?Reason:", re.S)

def clean_text(t):
    # Phase-1 cleaning: stamps out, special chars -> ASCII, drop other non-ASCII, single spaces
    t = SIG_RE.sub(" ", t)
    t = "".join(CHAR_MAP.get(c, c) for c in t)
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", t).strip()

cache = OUT / "token_counts.json"
TOKS = read_json(cache) if cache.exists() else None
if TOKS is not None and all(len(TOKS.get(n, [])) == len(SPLITS[n]) for n in SPLITS):
    print("loaded cached token counts")
else:
    TOKS = {}
    for name, docs in SPLITS.items():
        counts = []
        for i in tqdm(range(0, len(docs), 256), desc=f"tokenising {name}"):
            batch = [" " + clean_text(d["text"]) for d in docs[i:i + 256]]
            counts += [len(e.ids) for e in tok.encode_batch(batch, add_special_tokens=False)]
        TOKS[name] = counts
    cache.write_text(json.dumps(TOKS))

def n_windows(n, Lw=1536, stride=1024):
    return 1 if n <= Lw else 1 + int(np.ceil((n - Lw) / stride))

tok_rows = {}
for name, t in TOKS.items():
    t = np.asarray(t)
    tok_rows[name] = {**pct_row(t), "tok/word": round(t.sum() / max(sum(WORDS[name]), 1), 3),
                      "<=512 %": round(100 * (t <= 512).mean(), 1), "<=1536 %": round(100 * (t <= 1536).mean(), 1),
                      "<=3072 %": round(100 * (t <= 3072).mean(), 1), "<=6144 %": round(100 * (t <= 6144).mean(), 1),
                      "windows/doc": round(float(np.mean([n_windows(x) for x in t])), 2),
                      "total M tokens": round(t.sum() / 1e6, 1)}
display(pd.DataFrame(tok_rows).T.rename_axis("bpe32k tokens per document"))

TRAIN_TPS, INFER_TPS = 26_000, 90_000   # tokens/s, measured in Phase 4 (A2 full FT, micro-batch 2 x 1536)
def tokens_read(n, strategy):
    # how many tokens the model reads for one case of n tokens
    if strategy == "whole case (all windows)":       # overlapping windows -> reads more than n
        return n if n <= 1536 else 1536 * n_windows(n)
    if strategy == "512 tokens":
        return min(n, 512)
    return min(n, 1536)                               # first 1536 / last 1536 / first 768 + last 768

STRATEGIES = ["whole case (all windows)", "1536 tokens (first / last / first+last)", "512 tokens"]
est = {}
for s in STRATEGIES:
    tr_tok = sum(tokens_read(n, s) for n in TOKS["train"])
    ev_tok = sum(tokens_read(n, s) for sp in ("dev", "test") for n in TOKS[sp])
    est[s] = {"M tokens read / epoch": round(tr_tok / 1e6, 1),
              "train min / epoch (full FT)": round(tr_tok / TRAIN_TPS / 60, 1),
              "dev+test inference min": round(ev_tok / INFER_TPS / 60, 1)}
est = pd.DataFrame(est).T
display(est.rename_axis("reading strategy (train split)"))

# do longer cases have more statutes?
k_tr = np.array([len(d["labels"]) for d in SPLITS["train"]])
q = pd.qcut(np.asarray(TOKS["train"]), 4, labels=["Q1 short", "Q2", "Q3", "Q4 long"], duplicates="drop")
display(pd.DataFrame({"length quartile": q, "labels/case": k_tr}).groupby("length quartile", observed=True)["labels/case"]
        .agg(["count", "mean"]).round(2))
print(f"Spearman(length, number of labels) = {spearman(TOKS['train'], k_tr):.3f}")

tokenizer: data\tokenizer\bpe32k.json | vocab 32000


tokenising test: 100%|██████████| 51/51 [00:16<00:00,  3.07it/s]


,mean,p10,p50,p90,p99,max,tok/word,<=512 %,<=1536 %,<=3072 %,<=6144 %,windows/doc,total M tokens
bpe32k tokens per document,,,,,,,,,,,,,
train,1713.0,151.0,646.0,4180.0,12325.0,141120.0,1.524,42.9,69.0,83.8,95.3,2.04,73.2
dev,1913.1,151.0,808.0,4504.0,13546.0,117095.0,1.532,37.9,64.2,81.1,94.6,2.19,19.5
test,1860.3,153.0,799.0,4298.0,13170.0,222314.0,1.535,38.0,65.3,82.6,95.0,2.14,24.2


,M tokens read / epoch,train min / epoch (full FT),dev+test inference min
reading strategy (train split),,,
whole case (all windows),103.1,66.1,11.5
1536 tokens (first / last / first+last),35.1,22.5,3.8
512 tokens,17.3,11.1,1.8


,count,mean
length quartile,,
Q1 short,10715,3.72
Q2,10666,3.50
Q3,10684,3.56
Q4 long,10685,4.04


Spearman(length, number of labels) = 0.061


## 10. Are statutes named in the text? Where?

If the cases still say _"... under Section 302 of the Indian Penal Code ..."_, the statute is almost written in the input
and a model only has to find it; if the mentions were masked (replaced by a placeholder), the model must infer the
statute from the facts. We count:

- mentions like `section 302`, `s. 302`, `article 21`, `act`, `code`, and placeholder-looking tokens (`<...>`, `[...]`, all-caps words with `_`)
- **where** the `section N` mentions are (position in the case, in tenths) -> is there a part of the case worth selecting?
- for the most frequent statute IDs: which `section N` string is most typical for cases with that ID
  (lift = how much more often it appears in those cases than in all cases). A strong match means the ID can be read off the text.


In [12]:
SEC_RE = re.compile(r"\b(?:section|sec\.|s\.|u/s\.?|under section)\s*(\d{1,4}[a-z]?)\b", re.I)
PATTERNS = {"'section N' style mention": SEC_RE,
            "'article N'": re.compile(r"\barticles?\s+\d{1,3}", re.I),
            "'penal code' / 'ipc'": re.compile(r"\bpenal code\b|\bi\.?\s?p\.?\s?c\b", re.I),
            "'code of criminal procedure' / 'crpc'": re.compile(r"criminal procedure|\bcr\.?\s?p\.?\s?c\b", re.I),
            "'... act'": re.compile(r"\b[a-z]+ act\b", re.I),
            "<placeholder>": re.compile(r"<[^<>\s]{2,30}>"),
            "[placeholder]": re.compile(r"\[[^\[\]]{2,30}\]"),
            "ALL_CAPS_WITH_UNDERSCORE": re.compile(r"\b[A-Z]+_[A-Z_]+\b")}

mention_cache = OUT / "mentions.json"
if mention_cache.exists():
    MENT = read_json(mention_cache)
    print("loaded cached mention analysis")
else:
    MENT = {"share": {}, "per_doc": {}, "positions": [0] * 10, "placeholders": {}, "sections_per_doc": {}}
    ph = collections.Counter()
    for name in ("train", "dev", "test"):
        has = collections.Counter(); n_sec = []
        sec_sets = []
        for d in tqdm(SPLITS[name], desc=f"mentions {name}", leave=False):
            t = d["text"]
            for k, rx in PATTERNS.items():
                if rx.search(t):
                    has[k] += 1
            secs = [(m.start(), m.group(1).lower()) for m in SEC_RE.finditer(t)]
            n_sec.append(len(secs))
            if name == "train":
                for pos, _ in secs:
                    MENT["positions"][min(9, int(10 * pos / max(len(t), 1)))] += 1
                for rx in (PATTERNS["<placeholder>"], PATTERNS["[placeholder]"], PATTERNS["ALL_CAPS_WITH_UNDERSCORE"]):
                    ph.update(m.group(0) for m in rx.finditer(t[:50000]))
            sec_sets.append(sorted({s for _, s in secs}))
        MENT["share"][name] = {k: round(100 * has[k] / len(SPLITS[name]), 1) for k in PATTERNS}
        MENT["per_doc"][name] = pct_row(n_sec) if n_sec else {}
        MENT["sections_per_doc"][name] = sec_sets
    MENT["placeholders"] = ph.most_common(30)
    mention_cache.write_text(json.dumps(MENT))

display(pd.DataFrame(MENT["share"]).rename_axis("% of cases containing"))
display(pd.DataFrame(MENT["per_doc"]).T.rename_axis("'section N' mentions per case"))
pos = np.array(MENT["positions"], float)
print("where 'section N' mentions are (train, share per tenth of the case):",
      " ".join(f"{100 * x / max(pos.sum(), 1):.0f}%" for x in pos))
print("most common placeholder-like tokens:", MENT["placeholders"][:15])

# which 'section N' string is most typical for each frequent statute ID?
sec_tr = MENT["sections_per_doc"]["train"]
sec_df = collections.Counter(s for ss in sec_tr for s in ss)
link = []
for l in order[:20]:
    idx = [i for i, d in enumerate(SPLITS["train"]) if l in d["labels"]]
    c = collections.Counter(s for i in idx for s in sec_tr[i])
    best = sorted(((c[s] / len(idx)) / (sec_df[s] / tr_n), s) for s in c if c[s] >= max(5, 0.1 * len(idx)))
    if best:
        lift, s = best[-1]
        link.append({"statute ID": l, "train cases": len(idx), "most typical mention": f"section {s}",
                     "% of its cases with it": round(100 * c[s] / len(idx), 1),
                     "% of all cases with it": round(100 * sec_df[s] / tr_n, 1), "lift": round(lift, 1)})
    else:
        link.append({"statute ID": l, "train cases": len(idx), "most typical mention": "-"})
print("frequent statute IDs and their most typical 'section N' mention (train):")
display(pd.DataFrame(link))

,train,dev,test
% of cases containing,,,
'section N' style mention,2.5,3.3,3.5
'article N',0.4,0.8,0.6
'penal code' / 'ipc',1.9,2.8,3.0
'code of criminal procedure' / 'crpc',0.9,1.1,1.0
'... act',15.6,17.4,17.2
<placeholder>,99.1,99.1,99.0
[placeholder],4.3,5.3,5.2
ALL_CAPS_WITH_UNDERSCORE,0.0,0.0,0.0


,mean,p10,p50,p90,p99,max
'section N' mentions per case,,,,,,
train,0.0,0.0,0.0,0.0,1.0,25.0
dev,0.1,0.0,0.0,0.0,1.0,25.0
test,0.1,0.0,0.0,0.0,2.0,91.0


where 'section N' mentions are (train, share per tenth of the case): 13% 18% 17% 15% 9% 5% 6% 5% 5% 7%
most common placeholder-like tokens: [('<ENTITY>', 2205625), ('<ACT>', 218303), ('<SECTION>', 218128), ('</span>', 429), ('</font>', 394), ('[<ENTITY>]', 317), ('[P.W.1]', 224), ('</p>', 213), ('[P.W.2]', 186), ('[A1]', 185), ('[PW-1]', 182), ('[A3]', 143), ('[A-1]', 141), ('[Ex.P1]', 134), ('[PW2]', 129)]
frequent statute IDs and their most typical 'section N' mention (train):


,statute ID,train cases,most typical mention
0,5,11730,-
1,34,10380,-
2,97,7880,-
3,43,7728,-
4,39,7120,-
5,12,4324,-
6,13,4136,-
7,10,4134,-
8,77,4046,-
9,94,3835,-


## 11. Which part of the case carries the signal? (word-based Naive Bayes, cached)

Before spending GPU hours, a cheap model tells us which part of a case points to the statutes.

**Binary relevance**: one separate yes/no classifier per statute (the simplest multi-label method; our neural models will
also make one independent sigmoid decision per statute).

**Binarised Naive Bayes** per statute (same idea as Phase 4), explained simply:

- each document = the **set** of words in it (lower-case, letters and digits, words seen in >= 5 training cases)
- for statute `l` and word `w`: `r[l, w] = log( P(w | cases with l) / P(w | cases without l) )`
  (positive = the word is typical for cases with this statute)
- score of a case = sum of `r[l, w]` over its words + log(prior odds of `l`)
- two decision rules: **score > 0**, and **top-k** (the k highest-scoring statutes, k = average labels per case)

Same classifier, four inputs: **first 512 words**, **last 512 words**, **first 256 + last 256**, **whole case**.
Trained on train, scored on dev. Analysis only, not a submission (Phase 5 must use my Phase-1 models).


In [13]:
WORD_RE = re.compile(r"[a-z]+|\d+[a-z]?")
PARTS = ["first 512", "last 512", "first 256 + last 256", "whole case"]

def section_words(t, part):
    w = t.split()
    if part == "first 512": w = w[:512]
    elif part == "last 512": w = w[-512:]
    elif part == "first 256 + last 256": w = w[:256] + w[-256:] if len(w) > 512 else w
    return set(WORD_RE.findall(" ".join(w).lower()))

LID = {l: i for i, l in enumerate(ALL_IDS)}          # statute ID -> row in the count matrix

def train_nb_multilabel(docs, part, min_df=5, alpha=1.0):
    # pass 1: document frequency of every word -> vocabulary
    df = collections.Counter()
    for d in tqdm(docs, desc=f"  {part}: vocabulary", leave=False):
        df.update(section_words(d["text"], part))
    vocab = {w: i for i, w in enumerate(w for w, c in df.items() if c >= min_df)}
    V = len(vocab)
    # pass 2: for every statute, in how many of its cases each word appears
    pos = np.zeros((L, V), np.float32); df_all = np.zeros(V, np.float32); n_pos = np.zeros(L)
    for d in tqdm(docs, desc=f"  {part}: counts", leave=False):
        idx = np.fromiter((vocab[w] for w in section_words(d["text"], part) if w in vocab), np.int64)
        df_all[idx] += 1
        for l in d["labels"]:
            pos[LID[l], idx] += 1
            n_pos[LID[l]] += 1
    neg = df_all[None, :] - pos                         # counts in the cases WITHOUT the statute
    p_pos = (pos + alpha) / (pos.sum(1, keepdims=True) + alpha * V)
    p_neg = (neg + alpha) / (neg.sum(1, keepdims=True) + alpha * V)
    R = np.log(p_pos) - np.log(p_neg)                   # r[l, w]
    n = len(docs)
    prior = np.log((n_pos + alpha) / (n - n_pos + alpha))
    return vocab, R.astype(np.float32), prior

def nb_scores(model, docs, part):
    vocab, R, prior = model
    S = np.zeros((len(docs), L), np.float32)
    for i, d in enumerate(docs):
        idx = [vocab[w] for w in section_words(d["text"], part) if w in vocab]
        S[i] = R[:, idx].sum(1) + prior
    return S

def decide(S, rule, k=1):
    if rule == "score > 0":
        return [[ALL_IDS[j] for j in np.flatnonzero(s > 0)] for s in S]
    return [[ALL_IDS[j] for j in np.argsort(-s)[:k]] for s in S]           # top-k

nb_cache = OUT / "nb_sections.json"
if nb_cache.exists():
    NB = read_json(nb_cache)
    print("loaded cached Naive Bayes results")
else:
    NB = {"results": {}, "top_words": {}, "per_label_whole": {}}
    for part in tqdm(PARTS, desc="sections"):
        model = train_nb_multilabel(SPLITS["train"], part)
        S = nb_scores(model, SPLITS["dev"], part)
        for rule in ("score > 0", f"top-{k_med}"):
            r = lsi_scores(gold_dev, decide(S, rule, k_med))
            NB["results"][f"{part} | {rule}"] = {m: v for m, v in r.items() if m != "per_label"}
            if part == "whole case" and rule == "score > 0":
                NB["per_label_whole"] = {str(l): round(v, 4) for l, v in r["per_label"].items()}
        if part == "whole case":                       # most typical words for the 15 most frequent statutes
            vocab, R, _ = model
            inv = np.array(list(vocab.keys()))
            for l in order[:15]:
                NB["top_words"][str(l)] = inv[np.argsort(-R[LID[l]])[:12]].tolist()
    nb_cache.write_text(json.dumps(NB, indent=1))

display(pd.DataFrame(NB["results"]).T.rename_axis("Naive Bayes on ... (dev)"))

sections: 100%|██████████| 4/4 [02:12<00:00, 33.22s/it]


,macro_f1,macro_f1_union,micro_f1,micro_p,micro_r,subset_acc,labels/case predicted
Naive Bayes on ... (dev),,,,,,,
first 512 | score > 0,0.2392,0.2392,0.3326,0.2303,0.5983,0.0028,10.17
first 512 | top-4,0.2318,0.2318,0.3445,0.3409,0.3482,0.0032,4.00
last 512 | score > 0,0.2449,0.2449,0.3367,0.2357,0.5890,0.0038,9.79
last 512 | top-4,0.2340,0.2340,0.3427,0.3391,0.3464,0.0033,4.00
first 256 + last 256 | score > 0,0.2468,0.2468,0.3432,0.2418,0.5907,0.0033,9.56
first 256 + last 256 | top-4,0.2363,0.2363,0.3528,0.3491,0.3566,0.0031,4.00
whole case | score > 0,0.2226,0.2226,0.3187,0.2215,0.5681,0.0032,10.04
whole case | top-4,0.2185,0.2185,0.3490,0.3453,0.3528,0.0035,4.00


### Naive Bayes per statute: does frequency decide the F1? (whole case, score > 0)

F1 of each statute against the number of training cases it has. If rare statutes get F1 ≈ 0, rare-statute handling
(class weights, per-label thresholds, ...) will matter a lot for macro-F1.


In [14]:
pl = pd.DataFrame({"train cases": freq["train"], "dev cases": freq["dev"],
                   "NB F1 (whole case)": pd.Series({int(k): v for k, v in NB["per_label_whole"].items()})})
pl = pl[pl["dev cases"] > 0]
pl["bucket"] = pd.cut(pl["train cases"], [-1, 9, 29, 99, 299, 999, np.inf],
                      labels=["1-9", "10-29", "30-99", "100-299", "300-999", ">=1000"])
display(pl.groupby("bucket", observed=True)["NB F1 (whole case)"].agg(["count", "mean", "min", "max"]).round(3)
        .rename_axis("train cases per statute"))
print(f"Spearman(train frequency, NB F1) = {spearman(pl['train cases'], pl['NB F1 (whole case)'].fillna(0)):.3f}")

,count,mean,min,max
train cases per statute,,,,
100-299,2,0.025,0.000,0.050
300-999,55,0.149,0.011,0.498
>=1000,43,0.326,0.078,0.704


Spearman(train frequency, NB F1) = 0.704


## 12. What do the frequent statute IDs probably mean?

The 12 words most typical for each of the 15 most frequent statutes (from the whole-case Naive Bayes). We do not get
the statute names, but words like _murder, dowry, cheque_ tell us which law an ID probably is - useful for the error
analysis and the viva. (Rare names can appear because NB over-trusts rare words.)


In [15]:
tw = pd.DataFrame({"statute ID": [int(l) for l in NB["top_words"]],
                   "train cases": [int(freq.loc[int(l), "train"]) for l in NB["top_words"]],
                   "most typical words": [", ".join(w) for w in NB["top_words"].values()]})
with pd.option_context("display.max_colwidth", 200):
    display(tw)

,statute ID,train cases,most typical words
0,5,11730,"124225, landholdings, 124226, ajs, gannuar, impleadement, baduria, rassi, gangarampur, munde, 7817, cheaters"
1,34,10380,"wouldappear, theeyewitnesses, conductedinquest, bonedeep, homicidalviolence, andpanchayatdars, avoluntary, exhumed, objectsrecovered, pericardial, thebloodstained, obtuse"
2,97,7880,"thisquash, venkatachariar, 2013294, 2013341, criminallyintimidated, sinthamani, 11664, subbammal, tnhb, 8480, shead, namesand"
3,43,7728,"sngh, thissuggestion, 12709, odtinformant, 202c, hasurs, petitionersand, 5654, 130o, 11664, raigadh, ves"
4,39,7120,"approbates, sheltering, flutter, congressmen, badmashi, hamdast, hooliganism, 4190, jhalra, tanna, 9982, kuria"
5,12,4324,"blowwith, sardaron, mobsters, limkheda, chahida, doob, concernedsessions, velammal, jalta, saap, keshdhari, tabbals"
6,13,4136,"apparao, eraser, blowwith, limkheda, mobsters, ofunlawful, sngh, sikhon, thissuggestion, ander, aboutto, chahida"
7,10,4134,"buildtech, landholdings, gannuar, 124225, 124226, ajs, impleadement, rasa, scu, lookingafter, limkheda, hamdast"
8,77,4046,"124226, ajs, impleadement, gannuar, landholdings, 124225, pustika, abn, adjoured, antvyavsayi, amro, koylighansi"
9,94,3835,"simplest, unfed, ofdowry, starve, chuchak, fulfillling, 11664, orharassment, nagpanchami, forcohabitation, tomatrimonial, 498a"


## 13. Save summary


In [16]:
summary = {
    "format": FORMAT, "test_had_labels": TEST_HAD_LABELS,
    "checks": checks.reset_index().to_dict(orient="records"),
    "duplicate_groups": len(dup_groups), "duplicate_split_pairs": {"&".join(k): v for k, v in pair_counts.items()},
    "conflicting_duplicates": len(conflicting), "ids_shared": ids_shared,
    "n_labels": L, "label_min": min(ALL_IDS), "label_max": max(ALL_IDS), "ids_missing_in_range": len(missing),
    "dev_only_labels": dev_only, "train_only_labels": train_only,
    "cardinality": card, "frequency_buckets": brows,
    "label_frequency": {int(l): {"train": int(r["train"]), "dev": int(r["dev"])} for l, r in freq.iterrows()},
    "label_sets_train": len(sets), "dev_sets_seen_in_train": round(float(dev_sets_seen), 4),
    "top_pairs": prow, "baselines_dev": base_tbl.to_dict(orient="index"),
    "style": {n: dict(c) for n, c in STYLE.items()},
    "token_stats": tok_rows, "epoch_estimates": est.to_dict(orient="index"),
    "mentions_share": MENT["share"], "section_mentions_per_doc": MENT["per_doc"],
    "section_mention_positions": MENT["positions"], "statute_to_mention": link,
    "naive_bayes_dev": NB["results"], "nb_top_words": NB["top_words"],
    "tokenizer": str(TOK_PATH.relative_to(ROOT)),
}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print("saved", (OUT / "data_summary.json").relative_to(ROOT))

saved experiments\phase5\p5_01_data\data_summary.json
